## **Introduction to Integer Programming and Applications with Julia**

<table>
  <tr>
    <td>Chapter</td>
    <td>3 - Location Problems</td>
  </tr>
  <tr>
    <td>Section</td>
    <td>3.1.2.1 - Solving the PMP-C</td>
  </tr>
  <tr>
    <td>Author(s)</td>
    <td>Luiz Henrique Nogueira Lorena</td>
  </tr>
</table>

In [3]:
using JuMP      # Modeling language
using HiGHS     # Solver
using CSV       # Data handling
using Distances # Distance computations

# Include utility functions for plotting the solution
include("utils/pmp-c_utils.jl")

# Load latitude and longitude for facilities and clients
clients_coordinates = CSV.read("data/pmp_clients.csv", CSV.Tables.matrix, header=false)
facilities_coordinates = CSV.read("data/pmp_facilities.csv", CSV.Tables.matrix, header=false)

# Problem dimensions
n = size(clients_coordinates, 1)    # Number of clients
m = size(facilities_coordinates, 1) # Number of facilities

# Use Haversine distance to compute distance matrix
D = Distances.pairwise(Distances.Haversine(), clients_coordinates, facilities_coordinates, dims=1)

# Number of facilities to select
p = 3

# Create the model
model = JuMP.Model(HiGHS.Optimizer)

# Silent mode (solver output is not printed)
JuMP.set_silent(model)

# Decision variables
@variable(model, y[1:m], Bin)
@variable(model, x[1:n, 1:m], Bin)

# Objective: minimize total distance
@objective(model, Min, sum(D[i, j] * x[i, j] for i in 1:n, j in 1:m))

# Constraint: exactly p facilities selected
@constraint(model, sum(y) == p)

# Constraint: each client assigned to one facility
@constraint(model, [i in 1:n], sum(x[i, j] for j in 1:m) == 1)

# Constraint: assignment only to open facilities
@constraint(model, [i in 1:n, j in 1:m], x[i, j] <= y[j])

# Solve the model
JuMP.optimize!(model)

# Extract solution
y_value = JuMP.value.(y)
x_value = JuMP.value.(x)
objective_value = JuMP.objective_value(model)

# Print objective value
println("Objective value: $objective_value meters")

# Print selected facilities
facility_ids = findall(y_value .> 0.5)
println("Selected Facilities (p=$p): $facility_ids")

# Print total client assignments for each selected facility
assignments = Dict()
for facility_id in facility_ids
    clients = findall(x_value[:, facility_id] .> 0.5)
    assignments[facility_id] = clients
    println("Facility: $facility_id | $(length(clients)) clients assigned")
end

# Plot the solution
map = plot_solution(clients_coordinates, facilities_coordinates, assignments, p)
display(map)

Objective value: 20591.95914310779 meters
Selected Facilities (p=3): [5, 7, 8]
Facility: 5 | 43 clients assigned
Facility: 7 | 16 clients assigned
Facility: 8 | 33 clients assigned


Python: <folium.folium.Map object at 0x7fde8469c7d0>

<table>
  <tr>
    <td>Chapter</td>
    <td>3 - Location Problems</td>
  </tr>
  <tr>
    <td>Section</td>
    <td>3.1.3.1 - Solving the PMP-N</td>
  </tr>
  <tr>
    <td>Author(s)</td>
    <td>Luiz Henrique Nogueira Lorena</td>
  </tr>
</table>

In [4]:
using JuMP      # Modeling language
using HiGHS     # Solver
using CSV       # Data handling
using Distances # Distance computations

# Include utility functions for plotting the solution
include("utils/pmp-n_utils.jl")

# Load latitude and longitude for all points
coordinates = CSV.read("data/coordinates.csv", CSV.Tables.matrix, header=false)

# Problem dimensions
n = size(coordinates, 1) # Number of points

# Use Haversine distance to compute distance matrix
D = Distances.pairwise(Distances.Haversine(), coordinates, dims=1)

# Number of facilities to select
p = 3

# Create the model
model = JuMP.Model(HiGHS.Optimizer)

# Silent mode (solver output is not printed)
JuMP.set_silent(model)

# Decision variables
@variable(model, x[1:n, 1:n], Bin)

# Objective: minimize total distance
@objective(model, Min, sum(D[i, j] * x[i, j] for i in 1:n, j in 1:n))

# Constraint: exactly p facilities selected
@constraint(model, sum(x[j, j] for j in 1:n) == p)

# Constraint: each client assigned to one facility
@constraint(model, [i in 1:n], sum(x[i, j] for j in 1:n) == 1)

# Constraint: assignment only to open facilities
@constraint(model, [i in 1:n, j in 1:n], x[i, j] <= x[j, j])

# Solve the model
JuMP.optimize!(model)

# Extract solution
x_value = JuMP.value.(x)
objective_value = JuMP.objective_value(model)

# Print objective value
println("Objective value: $objective_value meters")

# Print selected facilities
facility_ids = findall(x_value[j, j] .> 0.5 for j in 1:n)
println("Selected Facilities (p=$p): $facility_ids")

# Print total client assignments for each selected facility
assignments = Dict()
for facility_id in facility_ids
    clients = findall(x_value[:, facility_id] .> 0.5)
    assignments[facility_id] = clients
    println("Facility: $facility_id | $(length(clients)) clients assigned")
end

# Plot the solution
map = plot_solution(coordinates, assignments, p)
display(map)

Objective value: 19211.026718733905 meters
Selected Facilities (p=3): [81, 92, 99]
Facility: 81 | 44 clients assigned
Facility: 92 | 26 clients assigned
Facility: 99 | 32 clients assigned


Python: <folium.folium.Map object at 0x7fde843b5e50>

<table>
  <tr>
    <td>Chapter</td>
    <td>3 - Location Problems</td>
  </tr>
  <tr>
    <td>Section</td>
    <td>3.2.1.2 - Solving the SCLP</td>
  </tr>
  <tr>
    <td>Author(s)</td>
    <td>Luiz Henrique Nogueira Lorena</td>
  </tr>
</table>

In [5]:
using JuMP  # Modeling language
using HiGHS # Solver
using CSV   # For reading CSV files

# Include utility functions for plotting the solution
include("utils/sclp_utils.jl")

# Load coordinates
coordinates = CSV.read("data/coordinates.csv", CSV.Tables.matrix, header=false)

# Create distance matrix
D = create_distance_matrix(coordinates)

# Define coverage radius (in meters)
radius = 200

# Create coverage matrix considering the defined radius
C = D .< radius

# Number of demand/facility points
n = size(D, 1)

# Create model
model = JuMP.Model(HiGHS.Optimizer)

# Silent mode (solver output is not printed)
JuMP.set_silent(model)

# Variables
@variable(model, x[1:n], Bin)

# Objective: minimize number of facilities opened
@objective(model, Min, sum(x))

# Constraint: each demand point is covered by at least one facility
@constraint(model, [i in 1:n], sum(C[i,j] * x[j] for j in 1:n) >= 1)

# Solve the model
JuMP.optimize!(model)

# Get solution vector
x_vals = JuMP.value.(x)

# Get indices of selected facilities
selected_facilities = findall(x_vals .> 0.5)

# Display solution
println("Number of facilities opened: ", Int(JuMP.objective_value(model)))
println("Selected facility indices: ", selected_facilities)

# Display solution on map
map = plot_solution(selected_facilities, coordinates, radius)
display(map)

Number of facilities opened: 11
Selected facility indices: [3, 10, 29, 30, 36, 47, 57, 58, 62, 88, 89]


Python: <folium.folium.Map object at 0x7fc0b47690f0>

<table>
  <tr>
    <td>Chapter</td>
    <td>3 - Location Problems</td>
  </tr>
  <tr>
    <td>Section</td>
    <td>3.2.2.2 - Solving the MCLP</td>
  </tr>
  <tr>
    <td>Author(s)</td>
    <td>Luiz Henrique Nogueira Lorena</td>
  </tr>
</table>

In [ ]:
using JuMP  # Modeling language
using HiGHS # Solver
using CSV   # For reading CSV files

# Include utility functions for plotting the solution
include("utils/mclp_utils.jl")

# Load coordinates
coordinates = CSV.read("data/coordinates.csv", CSV.Tables.matrix, header=false)

# Create distance matrix
D = create_distance_matrix(coordinates)

# Define coverage radius (in meters)
radius = 200 
    
# Number of facilities to select
p = 7

# Create coverage matrix considering the defined radius
C = D .< radius

# Number of demand/facility points
n = size(C, 1)

# Create a vector of equal demand for each point
d = ones(n)

# Create model
model = JuMP.Model(HiGHS.Optimizer)

# Silent mode (solver output is not printed)
JuMP.set_silent(model)

# Variables
@variable(model, x[1:n], Bin)
@variable(model, y[1:n], Bin)

# Objective: maximize covered demand
@objective(model, Max, sum(d[i]*y[i] for i in 1:n))

# Constraint: exactly p facilities are opened
@constraint(model, sum(x) == p)

# Constraint: each demand point is covered by at least one facility
@constraint(model, [i in 1:n], sum(C[i,j] * x[j] for j in 1:n) >= y[i])

# Solve the model
JuMP.optimize!(model)

# Get solution vectors
x_vals = JuMP.value.(x)
y_vals = JuMP.value.(y)

# Get indices of selected facilities
selected_facilities = findall(xi -> xi > 0.5, x_vals)
covered_points = findall(yi -> yi > 0.5, y_vals)

# Calculate coverage percentage
total_covered = length(covered_points)
coverage_percentage = round(100*total_covered/n, digits=2)

# Display solution
println("Facilities opened: $selected_facilities")
println("Points covered: $total_covered/$n ($coverage_percentage %)")

# Display solution on map
map = plot_solution(selected_facilities,covered_points, coordinates, radius)
display(map)

Facilities opened: [14, 28, 52, 59, 81, 82, 100]
Points covered: 90/102 (88.24 %)


Python: <folium.folium.Map object at 0x7fc0b40dac40>

<table>
  <tr>
    <td>Chapter</td>
    <td>3 - Location Problems</td>
  </tr>
  <tr>
    <td>Section</td>
    <td>3.2.2.2 - Solving the MCLP with random demands</td>
  </tr>
  <tr>
    <td>Author(s)</td>
    <td>Luiz Henrique Nogueira Lorena</td>
  </tr>
</table>

In [ ]:
using JuMP   # Modeling language
using HiGHS  # Solver
using CSV    # For reading CSV files
using Random # For generating random demands

# Seed to create random demand
Random.seed!(42)

# Include utility functions for plotting the solution
include("utils/mclp_utils.jl")

# Load coordinates
coordinates = CSV.read("data/coordinates.csv", CSV.Tables.matrix, header=false)

# Create distance matrix
D = create_distance_matrix(coordinates)

# Define coverage radius (in meters)
radius = 200 
    
# Number of facilities to select
p = 7

# Create coverage matrix considering the defined radius
C = D .< radius

# Number of demand/facility points
n = size(C, 1)

# Create a vector with a random demands
d = rand(n)

# Create model
model = JuMP.Model(HiGHS.Optimizer)

# Silent mode (solver output is not printed)
JuMP.set_silent(model)

# Variables
@variable(model, x[1:n], Bin)
@variable(model, y[1:n], Bin)

# Objective: maximize covered demand
@objective(model, Max, sum(d[i]*y[i] for i in 1:n))

# Constraint: exactly p facilities are opened
@constraint(model, sum(x) == p)

# Constraint: each demand point is covered by at least one facility
@constraint(model, [i in 1:n], sum(C[i,j] * x[j] for j in 1:n) >= y[i])

# Solve the model
JuMP.optimize!(model)

# Get solution vectors
x_vals = JuMP.value.(x)
y_vals = JuMP.value.(y)

# Get indices of selected facilities
selected_facilities = findall(xi -> xi > 0.5, x_vals)
covered_points = findall(yi -> yi > 0.5, y_vals)

# Calculate coverage percentage
total_covered = length(covered_points)
coverage_percentage = round(100*total_covered/n, digits=2)

# Display solution
println("Facilities opened: $selected_facilities")
println("Points covered: $total_covered/$n ($coverage_percentage %)")

# Display solution on map
map = plot_solution(selected_facilities,covered_points, coordinates, radius)
display(map)

Facilities opened: [14, 18, 47, 57, 59, 82, 98]
Points covered: 85/102 (83.33 %)


Python: <folium.folium.Map object at 0x7fc0957b96d0>